In [1]:
print("hii")

hii


In [4]:
! uv pip install scrapling langchain langchain-core langchain-openrouter python-dotenv
# ! uv pip scrapling install  # installs browser drivers

Using Python 3.13.3 environment at: D:\Langchain-dev\.venv
Checked 5 packages in 282ms


In [10]:
! uv pip install curl_cffi playwright browserforge

Using Python 3.13.3 environment at: D:\Langchain-dev\.venv
Resolved 16 packages in 471ms
Prepared 2 packages in 194ms
Installed 2 packages in 59ms
 + apify-fingerprint-datapoints==0.13.0
 + browserforge==1.2.4


In [11]:
"""
Funded Startup Tracker Agent
Sources: TechCrunch (funding news) + YC company directory
Stack  : Scrapling · LangChain · OpenRouter (nemotron)
"""

# ══════════════════════════════════════════════════════════════════════════════
# Imports
# ══════════════════════════════════════════════════════════════════════════════
import os
import json
import asyncio
from dotenv import load_dotenv
from pydantic import SecretStr

from scrapling.fetchers import AsyncFetcher

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_openrouter import ChatOpenRouter

load_dotenv()

openrouter_key = os.getenv("OPENROUTER_API_KEY")
assert openrouter_key, "OPENROUTER_API_KEY is not set in .env"

# ── Model ─────────────────────────────────────────────────────────────────────
model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    api_key=SecretStr(openrouter_key),
    temperature=0.6,
    max_tokens=1024,
)


# ══════════════════════════════════════════════════════════════════════════════
# Tools
# ══════════════════════════════════════════════════════════════════════════════

@tool
def scrape_techcrunch_funding(max_articles: int = 5) -> str:
    """
    Scrapes latest startup funding news from TechCrunch /category/venture/.
    Returns a JSON list of {title, url, summary} dicts.
    """
    async def _fetch():
        page = await AsyncFetcher.get(
            "https://techcrunch.com/category/venture/",
            stealthy_headers=True,
        )
        articles = []
        for card in page.css("article")[:max_articles]:
            title_el = card.css("h2 a, h3 a").get()
            if not title_el:
                continue
            title = str(getattr(title_el, "text", title_el)).strip()
            href  = getattr(title_el, "attrib", {}).get("href", "") if hasattr(title_el, "attrib") else ""
            blurb_el = card.css("p").get()
            blurb = str(getattr(blurb_el, "text", blurb_el)).strip() if blurb_el else ""
            articles.append({"title": title, "url": href, "summary": blurb})
        return articles

    results = asyncio.run(_fetch())
    return json.dumps(results, indent=2)


@tool
def scrape_yc_latest_batch(limit: int = 10) -> str:
    """
    Scrapes the YC company directory for the most recently added startups.
    Returns a JSON list of {name, description, batch, url} dicts.
    """
    async def _fetch():
        page = await AsyncFetcher.get(
            "https://www.ycombinator.com/companies?batch=S25",
            stealthy_headers=True,
        )
        companies = []
        # YC renders via React — grab og/meta or static fallback text blocks
        for card in page.css("._company_i9oky_355, [class*='_company']")[:limit]:
            name_el  = card.css("span[class*='_coName'], h3").get()
            desc_el  = card.css("span[class*='_coDescription'], p").get()
            name  = str(getattr(name_el, "text", name_el)).strip() if name_el else "N/A"
            desc  = str(getattr(desc_el, "text", desc_el)).strip() if desc_el else "N/A"
            link_el = card.css("a").get()
            href  = getattr(link_el, "attrib", {}).get("href", "") if link_el and hasattr(link_el, "attrib") else ""
            companies.append({
                "name": name,
                "description": desc,
                "batch": "S25",
                "url": f"https://www.ycombinator.com{href}" if href.startswith("/") else href,
            })
        return companies

    results = asyncio.run(_fetch())
    return json.dumps(results, indent=2)


@tool
def summarize_trends(raw_data: str) -> str:
    """
    Takes raw scraped startup data (JSON string) and returns a plain-English
    trend summary: top sectors, funding patterns, notable companies.
    """
    # This tool intentionally lets the LLM do the analysis —
    # the agent calls this after collecting data from the other two tools.
    return f"Analyze and summarize the following startup data:\n\n{raw_data}"


# ══════════════════════════════════════════════════════════════════════════════
# Agent Loop  (manual ReAct — works with any LangChain-compatible model)
# ══════════════════════════════════════════════════════════════════════════════

TOOLS = [scrape_techcrunch_funding, scrape_yc_latest_batch, summarize_trends]
TOOL_MAP = {t.name: t for t in TOOLS}

model_with_tools = model.bind_tools(TOOLS)

SYSTEM = (
    "You are a startup intelligence agent. "
    "When asked about funded startups or trends, use your tools in order:\n"
    "1. scrape_techcrunch_funding — get latest funding news\n"
    "2. scrape_yc_latest_batch   — get YC S25 batch startups\n"
    "3. summarize_trends         — synthesize both datasets into insights\n"
    "Always call tools before answering. Never make up startup data."
)


def run_agent(user_query: str, max_steps: int = 6) -> str:
    """Simple ReAct loop: call model → execute tools → repeat until done."""
    messages = [
        {"role": "system", "content": SYSTEM},
        HumanMessage(content=user_query),
    ]

    for step in range(max_steps):
        response = model_with_tools.invoke(messages)
        messages.append(response)

        # ── No tool calls → final answer ──────────────────────────────────────
        if not getattr(response, "tool_calls", None):
            return response.content if isinstance(response.content, str) else str(response.content)

        # ── Execute each tool call ─────────────────────────────────────────────
        for tc in response.tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]
            print(f"[step {step+1}] calling tool: {tool_name}({tool_args})")

            fn = TOOL_MAP.get(tool_name)
            if fn is None:
                result = f"Error: tool '{tool_name}' not found."
            else:
                try:
                    result = fn.invoke(tool_args)
                except Exception as e:
                    result = f"Tool error: {e}"

            messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )

    return "Max steps reached without a final answer."


# ══════════════════════════════════════════════════════════════════════════════
# Entry point
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    query = (
        "What are the latest funded startups and key trends "
        "from TechCrunch and Y Combinator S25 batch?"
    )
    print("\n" + "═" * 60)
    print("STARTUP TRACKER AGENT")
    print("═" * 60 + "\n")
    answer = run_agent(query)
    print("\n── FINAL ANSWER ──────────────────────────────────────────\n")
    print(answer)


════════════════════════════════════════════════════════════
STARTUP TRACKER AGENT
════════════════════════════════════════════════════════════

[step 1] calling tool: scrape_techcrunch_funding({'max_articles': 5})
[step 1] calling tool: scrape_yc_latest_batch({'limit': 10})

── FINAL ANSWER ──────────────────────────────────────────


